# GS Mapper — Photos to 3D Gaussian Splat (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/3dgs-robotics/blob/main/notebooks/photos_to_splat_colab.ipynb)

Turn a folder of photos into a **browser-viewable 3D Gaussian Splat** — pose-free (DUSt3R), no COLMAP.

**Runtime → Change runtime type → GPU (T4 is enough)**, then run the cells top to bottom.

Repo: https://github.com/rsasaki0109/3dgs-robotics · Live demo gallery: https://rsasaki0109.github.io/3dgs-robotics/splat.html

In [ ]:
#@title 1. Check the GPU
!nvidia-smi -L

In [ ]:
#@title 2. Install gs-mapper + DUSt3R (~3 min)
%cd /content
!git clone -q https://github.com/rsasaki0109/3dgs-robotics.git
%cd /content/3dgs-robotics
!pip -q install -e .
!pip -q install roma einops trimesh safetensors gsplat
!git clone -q --recursive --depth 1 https://github.com/naver/dust3r /tmp/dust3r
print("Setup done. Note: gsplat JIT-compiles its CUDA kernels on the first training step (a few extra minutes once).")

## 3. Get photos

**Option A** — try the bundled sample set right away (next cell).

**Option B** — upload your own: 8–16 photos orbiting one subject with ~70% overlap works best.

In [ ]:
#@title Option A: use the bundled sample photos
!mkdir -p /content/my_photos && cp docs/gallery/campus/*.jpg /content/my_photos/
!ls /content/my_photos

In [ ]:
#@title Option B: upload your own photos
import os
from google.colab import files

os.makedirs("/content/my_photos", exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f"/content/my_photos/{name}", "wb") as f:
        f.write(data)
print(f"{len(uploaded)} photos saved to /content/my_photos")

In [ ]:
#@title 4. Photos -> .splat (draft preset; ~5-15 min on T4)
# The DUSt3R checkpoint is fetched from the Hugging Face Hub automatically.
!gs-mapper photos-to-splat \
  --images /content/my_photos \
  --output /content/outputs/colab_splat \
  --quality draft \
  --dust3r-root /tmp/dust3r \
  --dust3r-checkpoint naver/DUSt3R_ViTLarge_BaseDecoder_512_dpt

In [ ]:
#@title 5. Download your splat and view it in the browser
from pathlib import Path
from google.colab import files

splat = next(Path("/content/outputs/colab_splat").glob("*.splat"))
print(f"{splat} ({splat.stat().st_size / 1e6:.1f} MB)")
files.download(str(splat))
print("\nNow open https://rsasaki0109.github.io/3dgs-robotics/splat.html and drag the downloaded .splat onto the page.")

## Next steps

- Cleaner results: `--quality balanced` or `--quality clean` (more frames + iterations).
- Metric-scale outdoor maps: `--preprocess mast3r` (see the [README](https://github.com/rsasaki0109/3dgs-robotics#readme)).
- Rosbags, external SLAM imports, and Physical AI scenario CI: [docs](https://rsasaki0109.github.io/3dgs-robotics/).

If this was fun, a ⭐ on [rsasaki0109/3dgs-robotics](https://github.com/rsasaki0109/3dgs-robotics) helps a lot!